In [ ]:
import pandas as pd

chapters = pd.read_csv("/content/chapters.csv")
interactions = pd.read_csv("/content/interactions.csv")

print(chapters.head())
print(interactions.head())

   chapter_id  chapter_sequence_no  book_id  author_id published_date  \
0     2812946                    1   139726      66847     1990-03-22   
1     4330764                    2   139726      66847     1990-04-09   
2     2664499                    3   139726      66847     1990-04-07   
3     2260666                    4   139726      66847     1990-05-18   
4     6069976                    1   191772      62262     2008-07-30   

                                       tags  
0                            Fantasy|Horror  
1      Fantasy|Young Adult|Literary Fiction  
2                                   Fantasy  
3                  Literary Fiction|Fantasy  
4  Horror|Young Adult|Romance|Graphic Novel  
        user_id  chapter_id  book_id
0  user_2378720     5894067   444295
1  user_2321122     2532511   785684
2  user_2335775     6777764   999595
3  user_7906001     7366896   748410
4  user_9981689     7853186   418083


In [ ]:


from collections import defaultdict


print("Chapters shape:", chapters.shape)
print("Interactions shape:", interactions.shape)


interactions = interactions.reset_index()
interactions = interactions.sort_values(["user_id", "index"])

chapter_to_book = dict(zip(chapters["chapter_id"], chapters["book_id"]))
chapter_to_seq = dict(zip(chapters["chapter_id"], chapters["chapter_sequence_no"]))

book_max_seq = chapters.groupby("book_id")["chapter_sequence_no"].max().to_dict()


user_sequences = interactions.groupby("user_id")["chapter_id"].apply(list)

user_books = interactions.groupby("user_id")["book_id"].apply(list)

def remove_duplicates(seq):
    res = []
    for x in seq:
        if not res or res[-1] != x:
            res.append(x)
    return res

user_books = user_books.apply(remove_duplicates)


chapters_sorted = chapters.sort_values(["book_id", "chapter_sequence_no"])

next_chapter_map = {}

for book_id, group in chapters_sorted.groupby("book_id"):
    ch_list = group["chapter_id"].tolist()
    for i in range(len(ch_list)-1):
        next_chapter_map[ch_list[i]] = ch_list[i+1]


transitions = defaultdict(lambda: defaultdict(int))

for seq in user_sequences:
    for i in range(len(seq)-1):
        curr = seq[i]
        nxt = seq[i+1]
        transitions[curr][nxt] += 1


book_transitions = defaultdict(lambda: defaultdict(int))

for seq in user_books:
    for i in range(len(seq)-1):
        curr = seq[i]
        nxt = seq[i+1]
        book_transitions[curr][nxt] += 1



chapters["tags_list"] = chapters["tags"].str.split("|")

book_genres = chapters.groupby("book_id")["tags_list"].sum()

def is_last_chapter(chapter_id):
    book = chapter_to_book[chapter_id]
    return chapter_to_seq[chapter_id] == book_max_seq[book]

def recommend_next_chapter(chapter_id):
    return next_chapter_map.get(chapter_id)

def recommend_from_sequence(chapter_id, k=5):
    next_items = transitions.get(chapter_id, {})
    sorted_items = sorted(next_items.items(), key=lambda x: -x[1])
    return [x[0] for x in sorted_items[:k]]

def recommend_next_book(book_id, k=5):
    next_books = book_transitions.get(book_id, {})
    sorted_books = sorted(next_books.items(), key=lambda x: -x[1])
    return [b for b, _ in sorted_books[:k]]

def recommend_by_genre(book_id, k=5):
    target_genres = set(book_genres.get(book_id, []))

    scores = []
    for b, genres in book_genres.items():
        if b == book_id:
            continue
        overlap = len(target_genres.intersection(set(genres)))
        if overlap > 0:
            scores.append((b, overlap))

    scores = sorted(scores, key=lambda x: -x[1])
    return [b for b, _ in scores[:k]]


def recommend(user_id, last_chapter):

    if not is_last_chapter(last_chapter):
        return {
            "type": "next_chapter",
            "recommendation": recommend_next_chapter(last_chapter)
        }

    current_book = chapter_to_book[last_chapter]

    seq_books = recommend_next_book(current_book)

    genre_books = recommend_by_genre(current_book)

    final_books = list(set(seq_books + genre_books))[:5]

    return {
        "type": "new_book",
        "recommendation": final_books
    }


popular_chapters = interactions["chapter_id"].value_counts().head(10).index.tolist()

def recommend_popular():
    return popular_chapters


def recall_at_k(actual, predicted, k=5):
    return int(actual in predicted[:k])


sample_user = list(user_sequences.keys())[0]
sample_seq = user_sequences[sample_user]


last_chapter = sample_seq[-1]

print("User:", sample_user)
print("Last Chapter:", last_chapter)

result = recommend(sample_user, last_chapter)

print("\nRecommendation Type:", result["type"])
print("Recommendations:", result["recommendation"])

Chapters shape: (50000, 6)
Interactions shape: (1000000, 3)
User: user_0000013
Last Chapter: 1958839

Recommendation Type: next_chapter
Recommendations: 5948621
